# MarceLLo - Full Training Pipeline

GRPO-based writing style capture. Trains an LLM to write like Marcelo.

**Pipeline:**
1. Load poems + blog posts + negative samples
2. Train style classifier (DeBERTa-v3-small) → reward model
3. GRPO training (Qwen2.5-1.5B + QLoRA) using classifier as reward
4. Evaluate

**Requirements:** Kaggle T4 GPU (15GB VRAM). Estimated total: ~2-3 hours.

---

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers>=4.40.0 datasets>=2.18.0 peft>=0.10.0 \
    trl>=0.15.0 accelerate>=0.28.0 scikit-learn>=1.4.0 \
    bitsandbytes>=0.43.0 rich>=13.0

# Prevent OOM fragmentation (the error Marcelo got before)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# ============================================================
# IMPORTS + VRAM MONITOR
# ============================================================
import torch
import gc
import re
import random
import numpy as np
from pathlib import Path
from datasets import Dataset


def vram_check(label=""):
    """Print VRAM usage with a status indicator."""
    if not torch.cuda.is_available():
        print("No GPU available!")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = total - allocated
    status = "OK" if free > 4.0 else "TIGHT" if free > 2.0 else "DANGER"
    bar_len = 30
    used_bars = int((allocated / total) * bar_len)
    bar = "#" * used_bars + "-" * (bar_len - used_bars)
    print(f"[VRAM {label}] [{bar}] {allocated:.1f}GB / {total:.1f}GB (free: {free:.1f}GB) {status}")


def vram_cleanup():
    """Force garbage collection and clear CUDA cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    vram_check("after cleanup")


print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU in Kaggle settings!'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.is_available() else "")
vram_check("initial")

---
## Phase 1: Data Pipeline

Load Marcelo's poems (poetic) + blog posts (standard) as positives.  
Load AI-written texts on the same topics as negatives.

In [ ]:
# ============================================================
# CLONE REPO + LOAD DATA
# ============================================================
!git clone https://github.com/marcelo-earth/marcello.git 2>/dev/null || echo "Already cloned"
REPO = Path("/kaggle/working/marcello")


def load_texts_from_dir(directory, min_length=50):
    """Load text samples from .txt files, split by paragraph."""
    texts = []
    for path in sorted(Path(directory).rglob("*.txt")):
        content = path.read_text(encoding="utf-8").strip()
        paragraphs = [p.strip() for p in content.split("\n\n") if p.strip()]
        for p in paragraphs:
            if len(p) >= min_length:
                texts.append(p)
    return texts


poems = load_texts_from_dir(REPO / "data/raw/writing_samples/")
blogs = load_texts_from_dir(REPO / "data/raw/writing_samples_blog/")
negatives = load_texts_from_dir(REPO / "data/raw/negative_samples/")

print(f"Poems (poetic):        {len(poems)} samples")
print(f"Blog posts (standard): {len(blogs)} samples")
print(f"Negatives (generic):   {len(negatives)} samples")
print(f"---")
print(f"Total positive: {len(poems) + len(blogs)}")
print(f"Total negative: {len(negatives)}")
print(f"Ratio: 1:{len(negatives) / (len(poems) + len(blogs)):.1f}")

In [ ]:
# ============================================================
# CREATE CONTRASTIVE DATASET
# ============================================================
random.seed(42)

positive_texts = poems + blogs
all_texts = positive_texts + negatives
all_labels = [1] * len(positive_texts) + [0] * len(negatives)

# Shuffle
combined = list(zip(all_texts, all_labels))
random.shuffle(combined)
texts, labels = zip(*combined)

full_dataset = Dataset.from_dict({"text": list(texts), "label": list(labels)})
split = full_dataset.train_test_split(test_size=0.15, seed=42)
train_ds = split["train"]
val_ds = split["test"]

print(f"Train: {len(train_ds)} ({sum(train_ds['label'])} pos, {len(train_ds) - sum(train_ds['label'])} neg)")
print(f"Val:   {len(val_ds)} ({sum(val_ds['label'])} pos, {len(val_ds) - sum(val_ds['label'])} neg)")
print(f"")
print(f"Sample positive: {positive_texts[0][:100]}...")
print(f"Sample negative: {negatives[0][:100]}...")

---
## Phase 2: Style Classifier (Reward Model)

Train DeBERTa-v3-small to distinguish Marcelo's writing from generic text.  
This becomes the reward signal for GRPO.

**Gate:** If accuracy < 80%, GRPO will get weak reward signals.

In [ ]:
# ============================================================
# TRAIN STYLE CLASSIFIER
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

CLF_MODEL = "microsoft/deberta-v3-small"

clf_tokenizer = AutoTokenizer.from_pretrained(CLF_MODEL)
clf_model = AutoModelForSequenceClassification.from_pretrained(CLF_MODEL, num_labels=2)
clf_model = clf_model.to("cuda")

vram_check("classifier loaded")


def tokenize(examples):
    return clf_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)


train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["text"])
train_tok.set_format("torch")
val_tok.set_format("torch")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


training_args = TrainingArguments(
    output_dir="outputs/classifier",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    fp16=False,
    report_to="none",
)

clf_trainer = Trainer(
    model=clf_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

clf_trainer.train()
vram_check("classifier trained")

In [ ]:
# ============================================================
# EVALUATE CLASSIFIER (ACCURACY GATE)
# ============================================================
results = clf_trainer.evaluate()

print(f"")
print(f"{'=' * 50}")
print(f"  STYLE CLASSIFIER RESULTS")
print(f"{'=' * 50}")
print(f"  Accuracy:  {results['eval_accuracy']:.4f}")
print(f"  Precision: {results['eval_precision']:.4f}")
print(f"  Recall:    {results['eval_recall']:.4f}")
print(f"  F1 Score:  {results['eval_f1']:.4f}")
print(f"{'=' * 50}")

GATE_PASSED = results["eval_accuracy"] >= 0.80

if GATE_PASSED:
    print(f"  PASS - Classifier is strong enough for GRPO")
else:
    print(f"  WARNING - Accuracy is low, GRPO reward signal will be noisy")
    print(f"  Consider adding more writing samples before proceeding")

print(f"{'=' * 50}")

# Save
clf_trainer.save_model("outputs/classifier/best")
clf_tokenizer.save_pretrained("outputs/classifier/best")
print("Classifier saved to outputs/classifier/best")

In [ ]:
# ============================================================
# VRAM CLEANUP: FREE CLASSIFIER TRAINER BEFORE GRPO
# ============================================================
# Delete the trainer (frees optimizer states and gradients)
# Keep clf_model reference - we'll reload a fresh one for reward
del clf_trainer
del clf_model
del train_tok
del val_tok
vram_cleanup()
print("Classifier trainer freed. Ready for GRPO.")

---
## Phase 3: GRPO Training

Load Qwen2.5-1.5B in **4-bit** (QLoRA) and train with GRPO.  
The style classifier scores each generation → group-relative advantages → policy update.

**VRAM budget:**  
- Base model (4-bit): ~1.5 GB  
- LoRA adapters + optimizer: ~2-3 GB  
- Reward classifier (fp16): ~0.3 GB  
- Activations/gradients: ~3-4 GB  
- **Total: ~7-9 GB** (T4 has 15 GB)

In [ ]:
# ============================================================
# LOAD BASE MODEL WITH QLoRA (4-BIT)
# ============================================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

BASE_MODEL = "Qwen/Qwen2.5-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

vram_check("base model loaded (4-bit)")

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
vram_check("LoRA applied")

In [ ]:
# ============================================================
# LOAD CLASSIFIER AS REWARD FUNCTION
# ============================================================
reward_clf = AutoModelForSequenceClassification.from_pretrained("outputs/classifier/best")
reward_tok = AutoTokenizer.from_pretrained("outputs/classifier/best")
reward_clf = reward_clf.to("cuda").half()
reward_clf.eval()

vram_check("reward classifier loaded")


def style_reward_fn(completions, **kwargs):
    """Score completions on how much they sound like Marcelo.
    
    Returns list of floats in [0, 1].
    1.0 = definitely Marcelo's style
    0.0 = generic/not Marcelo
    """
    # Debug first call to understand TRL's format
    if not hasattr(style_reward_fn, "_debugged"):
        print(f"[DEBUG] type(completions): {type(completions)}")
        print(f"[DEBUG] type(completions[0]): {type(completions[0])}")
        print(f"[DEBUG] completions[0]: {repr(completions[0])[:200]}")
        print(f"[DEBUG] kwargs keys: {list(kwargs.keys())}")
        style_reward_fn._debugged = True
    
    try:
        texts = []
        for c in completions:
            if isinstance(c, dict):
                c = c.get("content", str(c))
            elif isinstance(c, list):
                c = c[0]["content"] if isinstance(c[0], dict) else str(c[0])
            texts.append(str(c).strip() or "empty")
        
        with torch.no_grad():
            encoded = reward_tok(
                texts, padding=True, truncation=True, max_length=256, return_tensors="pt"
            ).to("cuda")
            outputs = reward_clf(**encoded)
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1]  # P(Marcelo)
        
        return probs.cpu().tolist()
    except Exception as e:
        print(f"[REWARD ERROR] {e}")
        import traceback
        traceback.print_exc()
        return [0.5] * len(completions)


# Quick sanity check
test_scores = style_reward_fn([
    "Permanezco en la frontera de la esperanza y la desilusión",
    "Research suggests that statistical analysis provides valuable insights.",
])
style_reward_fn._debugged = False  # Reset so GRPO first call also prints debug
print(f"Sanity check:")
print(f"  Marcelo-style text:  {test_scores[0]:.4f} (should be high)")
print(f"  Generic-style text:  {test_scores[1]:.4f} (should be low)")

In [ ]:
# ============================================================
# PREPARE PROMPTS FOR GRPO
# ============================================================
# Extract first sentences from Marcelo's writing as prompts
prompts = []
for text in positive_texts:
    match = re.match(r"^(.+?[.!?])\s", text)
    if match:
        prompt = match.group(1).strip()
        if 10 < len(prompt) < 200:
            prompts.append(prompt)

# Add some open-ended prompts in both languages
extra_prompts = [
    "Las estrellas de esta noche me recuerdan",
    "No espero una estrella que represente",
    "El tiempo es claramente",
    "Sueña.",
    "Imagina por un momento que puedes",
    "Somos la cuna de las preguntas,",
    "I've been thinking about what it means to truly learn.",
    "The most important thing about building something is",
    "Distribution matters more than",
    "Curiosity doesn't arise from nothing, because",
    "The future of education depends on",
    "Love as a design constraint means",
]
prompts.extend(extra_prompts)

# Deduplicate
prompts = list(dict.fromkeys(prompts))

prompt_dataset = Dataset.from_dict({"prompt": prompts})
print(f"Total prompts for GRPO: {len(prompt_dataset)}")
print(f"\nExamples:")
for p in prompts[:5]:
    print(f"  - {p[:80]}")

In [ ]:
# ============================================================
# GRPO TRAINING
# ============================================================
from trl import GRPOConfig, GRPOTrainer

vram_check("before GRPO setup")

grpo_config = GRPOConfig(
    output_dir="outputs/grpo",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-7,
    max_grad_norm=1.0,
    fp16=False,

    # GRPO specific
    num_generations=4,            # G: completions per prompt
    max_completion_length=128,    # tokens per completion
    temperature=0.8,

    # Memory saving
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Logging
    logging_steps=1,
    log_completions=True,
    save_steps=50,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=style_reward_fn,
    processing_class=tokenizer,
)

vram_check("GRPO trainer ready")
print("\nStarting GRPO training...")
print(f"  Prompts: {len(prompt_dataset)}")
print(f"  Generations per prompt: {grpo_config.num_generations}")
print(f"  Epochs: {grpo_config.num_train_epochs}")
print(f"  Effective batch size: {grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps}")
print()

In [ ]:
# ============================================================
# RUN GRPO TRAINING (this takes ~1-2 hours on T4)
# ============================================================
grpo_trainer.train()

vram_check("GRPO training complete")

# Save the LoRA adapter + tokenizer
model.save_pretrained("outputs/grpo/final")
tokenizer.save_pretrained("outputs/grpo/final")
print("\nModel saved to outputs/grpo/final")

---
## Phase 4: Evaluation

Generate text from the GRPO-trained model and score it with the classifier.

In [ ]:
# ============================================================
# GENERATE FROM MARCELLO
# ============================================================
from transformers import pipeline as hf_pipeline

# Clean up GRPO trainer to free VRAM
del grpo_trainer
vram_cleanup()

pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.8,
    do_sample=True,
    top_p=0.95,
)

eval_prompts = [
    "Las estrellas de esta noche me recuerdan",
    "No espero una estrella que represente",
    "I've been thinking about what it means to learn",
    "El tiempo es algo que",
    "The future of humanity depends on",
    "Creatividad es",
    "Somos nosotros, porque",
    "Distribution matters more than people think.",
]

print("=" * 60)
print("  MarceLLo - GRPO-Trained Generations")
print("=" * 60)

generated_texts = []
for prompt in eval_prompts:
    result = pipe(prompt, return_full_text=False)
    generated = result[0]["generated_text"].strip()
    generated_texts.append(generated)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {generated[:300]}")
    print("-" * 40)

In [ ]:
# ============================================================
# SCORE GENERATIONS WITH CLASSIFIER
# ============================================================
scores = style_reward_fn(generated_texts)
avg_score = sum(scores) / len(scores)

print(f"{'=' * 60}")
print(f"  STYLE SCORES (1.0 = sounds like Marcelo)")
print(f"{'=' * 60}")
for prompt, score in zip(eval_prompts, scores):
    bar = '#' * int(score * 30) + '-' * (30 - int(score * 30))
    print(f"  [{bar}] {score:.3f}  {prompt[:50]}")
print(f"{'=' * 60}")
print(f"  Average style score: {avg_score:.4f}")
print(f"{'=' * 60}")

if avg_score > 0.7:
    print("\n  MarceLLo has learned your style well!")
elif avg_score > 0.5:
    print("\n  Getting there - more data or more training epochs would help.")
else:
    print("\n  Needs more work. Try adding more writing samples.")

In [ ]:
# ============================================================
# INTERACTIVE: TRY YOUR OWN PROMPTS
# ============================================================
# Change the prompt below and re-run this cell!

PROMPT = "Pienso en el futuro y"

result = pipe(PROMPT, return_full_text=True, num_return_sequences=3)

print(f"Prompt: {PROMPT}\n")
for i, r in enumerate(result):
    text = r["generated_text"].strip()
    score = style_reward_fn([text[len(PROMPT):]])[0]
    print(f"--- Generation {i+1} (style: {score:.3f}) ---")
    print(text)
    print()

In [ ]:
# ============================================================
# DOWNLOAD THE MODEL (run this to save from Kaggle)
# ============================================================
# The LoRA adapter is small (~50MB), easy to download
import shutil

shutil.make_archive("/kaggle/working/marcello_model", "zip", "outputs/grpo/final")
print("Model saved as /kaggle/working/marcello_model.zip")
print("Download it from the Output tab!")

# Also save the classifier
shutil.make_archive("/kaggle/working/marcello_classifier", "zip", "outputs/classifier/best")
print("Classifier saved as /kaggle/working/marcello_classifier.zip")

vram_check("final")
print("\nDone! MarceLLo is trained.")